# ShadowCheck - Autonomous Zero-Day Triage Agent

This notebook implements a safe-by-default cybersecurity triage workflow for 2026 SOC teams:
- Structured threat output with `ThreatReport`
- Read-only recon tools (`check_running_versions`, `fetch_exploit_db`)
- Red-Team-Auditor prompt that tries to disprove exploitability
- Optional draft-only simulation payloads (never executed)
- Streaming-style UX via Gradio
- Observability/audit trail with Logfire

In [ ]:
# Optional install cell (uncomment if needed)
# %pip install -q pydantic pydantic-ai gradio requests packaging logfire python-dotenv

In [ ]:
from __future__ import annotations

import asyncio
import json
import os
import re
import socket
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import requests
from packaging import version
from pydantic import BaseModel, Field

try:
    import logfire
except Exception:
    logfire = None

try:
    from pydantic_ai import Agent, RunContext
except Exception as exc:
    raise RuntimeError(
        "pydantic-ai is required for this notebook. Install dependencies in the previous cell."
    ) from exc

## 1) Typed Security Output

The `ThreatReport` object is designed to feed downstream automations safely and predictably.

In [ ]:
class ThreatReport(BaseModel):
    cve_id: str = Field(description="CVE identifier such as CVE-2026-1234")
    severity_score: float = Field(ge=0, le=10, description="CVSS-like severity score")
    is_exploitable_locally: bool = Field(description="True if local environment appears exploitable")
    required_remediation: str = Field(description="Concrete action to remediate or mitigate")
    confidence_level: str = Field(description='One of: "Verified" | "Likely" | "Inferred"')


class ServiceFingerprint(BaseModel):
    host: str
    port: int
    open: bool
    banner: Optional[str] = None


class CVERecord(BaseModel):
    cve_id: str
    description: str
    cvss_score: float
    references: List[str] = Field(default_factory=list)
    vulnerable_version_range: Optional[str] = None


class TriageInput(BaseModel):
    repo_url: Optional[str] = None
    installed_packages: Dict[str, str] = Field(default_factory=dict)
    ports_to_scan: List[int] = Field(default_factory=lambda: [22, 80, 443, 8080, 8443])
    target_host: str = "127.0.0.1"

## 2) Dependency Injection for Agent Persona

`AgentDeps` carries environment context and an audit log object into tools.

In [ ]:
@dataclass
class AgentDeps:
    triage_input: TriageInput
    request_timeout_s: int = 12
    audit_events: List[Dict[str, Any]] = field(default_factory=list)

    def record(self, event: str, details: Dict[str, Any]) -> None:
        self.audit_events.append(
            {
                "ts": datetime.now(timezone.utc).isoformat(),
                "event": event,
                "details": details,
            }
        )


if logfire is not None:
    logfire.configure(service_name="shadowcheck")
    try:
        logfire.instrument_pydantic()
    except Exception:
        pass

## 3) Active Recon Tools (Read-Only)

These tools collect evidence only. They do not exploit or modify infrastructure.

In [ ]:
SYSTEM_PROMPT = """
You are ShadowCheck, a Red Team Auditor specialized in defensive triage.

Mission:
1) Attempt to disprove exploitability before asserting risk.
2) Prefer evidence from local environment checks over generic CVE text.
3) Mark confidence as:
   - Verified: direct evidence confirms exploitable path
   - Likely: strong indicators but one assumption remains
   - Inferred: weak indicators or missing local evidence
4) Never fabricate package versions, ports, or config states.
5) Recommend safe, practical remediation.

Output only valid ThreatReport.
"""


shadow_agent = Agent(
    model=os.getenv("SHADOWCHECK_MODEL", "openai:gpt-4o-mini"),
    deps_type=AgentDeps,
    output_type=ThreatReport,
    system_prompt=SYSTEM_PROMPT,
)


@shadow_agent.tool
async def check_running_versions(ctx: RunContext[AgentDeps]) -> List[ServiceFingerprint]:
    deps = ctx.deps
    host = deps.triage_input.target_host
    ports = deps.triage_input.ports_to_scan

    results: List[ServiceFingerprint] = []
    for port in ports:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(0.6)
        is_open = False
        banner = None
        try:
            is_open = sock.connect_ex((host, port)) == 0
            if is_open:
                try:
                    sock.sendall(b"HEAD / HTTP/1.0\r\n\r\n")
                    data = sock.recv(256)
                    banner = data.decode(errors="ignore").strip() or None
                except Exception:
                    banner = None
        finally:
            sock.close()

        results.append(
            ServiceFingerprint(
                host=host,
                port=port,
                open=is_open,
                banner=banner,
            )
        )

    deps.record("tool.check_running_versions", {"host": host, "ports": ports})
    return results


def _extract_cvss(metrics_block: Dict[str, Any]) -> float:
    candidates = [
        metrics_block.get("cvssMetricV31", []),
        metrics_block.get("cvssMetricV30", []),
        metrics_block.get("cvssMetricV2", []),
    ]
    for arr in candidates:
        if arr:
            cvss_data = arr[0].get("cvssData", {})
            score = cvss_data.get("baseScore")
            if isinstance(score, (int, float)):
                return float(score)
    return 0.0


@shadow_agent.tool
async def fetch_exploit_db(ctx: RunContext[AgentDeps], cve_id: str) -> CVERecord:
    deps = ctx.deps
    deps.record("tool.fetch_exploit_db.start", {"cve_id": cve_id})

    url = f"https://services.nvd.nist.gov/rest/json/cves/2.0?cveId={cve_id}"
    r = requests.get(url, timeout=deps.request_timeout_s)
    r.raise_for_status()
    payload = r.json()

    vulns = payload.get("vulnerabilities", [])
    if not vulns:
        return CVERecord(
            cve_id=cve_id,
            description="No vulnerability record found in source API.",
            cvss_score=0.0,
            references=[],
            vulnerable_version_range=None,
        )

    cve = vulns[0].get("cve", {})
    descriptions = cve.get("descriptions", [])
    description = next((d.get("value", "") for d in descriptions if d.get("lang") == "en"), "")
    refs = [r.get("url") for r in cve.get("references", []) if r.get("url")]
    metrics = cve.get("metrics", {})

    configurations = cve.get("configurations", [])
    vulnerable_range = None
    for config in configurations:
        for node in config.get("nodes", []):
            for cpe in node.get("cpeMatch", []):
                if cpe.get("vulnerable"):
                    start_inc = cpe.get("versionStartIncluding")
                    end_exc = cpe.get("versionEndExcluding")
                    start_exc = cpe.get("versionStartExcluding")
                    end_inc = cpe.get("versionEndIncluding")
                    pieces = [
                        f">={start_inc}" if start_inc else None,
                        f">{start_exc}" if start_exc else None,
                        f"<{end_exc}" if end_exc else None,
                        f"<={end_inc}" if end_inc else None,
                    ]
                    vulnerable_range = ",".join([p for p in pieces if p]) or None
                    break
            if vulnerable_range:
                break
        if vulnerable_range:
            break

    result = CVERecord(
        cve_id=cve_id,
        description=description,
        cvss_score=_extract_cvss(metrics),
        references=refs[:10],
        vulnerable_version_range=vulnerable_range,
    )

    deps.record("tool.fetch_exploit_db.done", {"cve_id": cve_id, "cvss": result.cvss_score})
    return result

## 4) Context Filter and Draft-Only Simulation

This logic prevents over-alerting by checking package-version plausibility before escalating.

In [ ]:
def version_in_range(installed: str, range_expr: Optional[str]) -> bool:
    if not range_expr:
        return True
    iv = version.parse(installed)
    checks = [x.strip() for x in range_expr.split(",") if x.strip()]
    for chk in checks:
        if chk.startswith(">=") and not (iv >= version.parse(chk[2:])):
            return False
        if chk.startswith(">") and not chk.startswith(">=") and not (iv > version.parse(chk[1:])):
            return False
        if chk.startswith("<=") and not (iv <= version.parse(chk[2:])):
            return False
        if chk.startswith("<") and not chk.startswith("<=") and not (iv < version.parse(chk[1:])):
            return False
    return True


def draft_simulation_command(cve_id: str, host: str, port: int) -> str:
    safe_cve = re.sub(r"[^A-Za-z0-9-]", "", cve_id)
    return (
        f"# Draft-only validation for {safe_cve}\n"
        f"curl -i --max-time 5 http://{host}:{port}/ \\n"
        f"  -H 'User-Agent: ShadowCheck-Validator/1.0'"
    )

## 5) Orchestration

`run_shadowcheck` prepares deps and calls the agent.

In [ ]:
def parse_pkg_text(pkg_text: str) -> Dict[str, str]:
    parsed: Dict[str, str] = {}
    for line in pkg_text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "==" in line:
            name, ver = line.split("==", 1)
            parsed[name.strip().lower()] = ver.strip()
    return parsed


async def run_shadowcheck(
    cve_id: str,
    pkg_text: str,
    host: str = "127.0.0.1",
    ports: Optional[List[int]] = None,
) -> Dict[str, Any]:
    triage_input = TriageInput(
        installed_packages=parse_pkg_text(pkg_text),
        target_host=host,
        ports_to_scan=ports or [22, 80, 443, 8080, 8443],
    )
    deps = AgentDeps(triage_input=triage_input)

    cve_data = await fetch_exploit_db.fn(RunContext(deps=deps), cve_id)
    fingerprints = await check_running_versions.fn(RunContext(deps=deps))

    likely_open_port = next((s.port for s in fingerprints if s.open), 80)
    simulation = draft_simulation_command(cve_id, host, likely_open_port)

    prompt = f"""
Assess exploitability for {cve_id}.

Evidence:
- CVE data: {cve_data.model_dump_json(indent=2)}
- Local service scan: {[s.model_dump() for s in fingerprints]}
- Installed packages: {triage_input.installed_packages}

Rules:
- Disprove exploitability before escalating.
- If no local evidence, reduce confidence.
- Keep remediation specific and actionable.
"""

    result = await shadow_agent.run(prompt, deps=deps)

    return {
        "threat_report": result.output.model_dump(),
        "draft_simulation": simulation,
        "audit_events": deps.audit_events,
        "service_fingerprints": [s.model_dump() for s in fingerprints],
        "cve_context": cve_data.model_dump(),
    }

## 6) Streaming Gradio SOC Console

The UI streams progress updates and ends with a structured JSON threat report.

In [ ]:
import gradio as gr


async def stream_shadowcheck(cve_id: str, package_text: str, host: str, ports_csv: str):
    try:
        ports = [int(p.strip()) for p in ports_csv.split(",") if p.strip()]
    except ValueError:
        yield "Invalid ports CSV. Example: 22,80,443,8080"
        return

    yield f"[1/4] Pulling CVE context for {cve_id}..."
    await asyncio.sleep(0.05)
    yield "[2/4] Performing read-only service fingerprint scan..."
    await asyncio.sleep(0.05)
    yield "[3/4] Running Red Team Auditor triage (disprove-first)..."

    result = await run_shadowcheck(cve_id=cve_id, pkg_text=package_text, host=host, ports=ports)

    yield "[4/4] Completed. Rendering structured report...\n\n" + json.dumps(result, indent=2)


def build_ui() -> gr.Blocks:
    with gr.Blocks(title="ShadowCheck SOC Console") as demo:
        gr.Markdown("# ShadowCheck SOC Console")
        gr.Markdown("Autonomous zero-day triage with typed outputs and audit trail.")

        cve_id = gr.Textbox(label="CVE ID", value="CVE-2024-3094")
        package_text = gr.Textbox(
            label="Installed Packages (pip style)",
            lines=8,
            value="openssl==3.0.2\nnginx==1.24.0",
        )
        host = gr.Textbox(label="Target Host", value="127.0.0.1")
        ports = gr.Textbox(label="Ports CSV", value="22,80,443,8080,8443")

        run_btn = gr.Button("Run Triage", variant="primary")
        output = gr.Textbox(label="Streaming Output", lines=24)

        run_btn.click(
            fn=stream_shadowcheck,
            inputs=[cve_id, package_text, host, ports],
            outputs=[output],
        )

    return demo

In [ ]:
# Launch UI (use share=True only if your network policy allows it)
# demo = build_ui()
# demo.launch(server_name="127.0.0.1", server_port=7860)

## 7) Quick Direct Test (Without UI)

In [ ]:
# Example direct run
# test_result = await run_shadowcheck(
#     cve_id="CVE-2024-3094",
#     pkg_text="openssl==3.0.2\nnginx==1.24.0",
#     host="127.0.0.1",
#     ports=[22, 80, 443, 8080],
# )
# print(json.dumps(test_result, indent=2))